# **Predicting Employee Retention**

### Objective
Develop a Logistic Regression model to predict binary outcomes (Stayed / Left) based on employee data.

### Business Objective
Help HR proactively identify employees likely to stay and understand the factors contributing to their loyalty.

## **1. Data Understanding**

### 1.0 Import Libraries

In [ ]:
# Suppress unnecessary warnings
import warnings
warnings.filterwarnings('ignore')

# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print('Libraries loaded successfully.')

### 1.1 Load the Data

In [ ]:
# Load the dataset  (utf-8-sig strips the UTF-8 BOM present in this file)
df = pd.read_csv('Employee_data.csv', encoding='utf-8-sig')
df.head()

In [ ]:
# Shape
df.shape

In [ ]:
# Column names
df.columns.tolist()

### 1.2 Basic Statistics

In [ ]:
df.describe()

### 1.3 Data Types & Null Overview

In [ ]:
df.info()

## **2. Data Cleaning**

### 2.1 Handle Missing Values

In [ ]:
# Count missing values per column
df.isnull().sum()

In [ ]:
# Percentage of missing values
round((df.isnull().sum() / len(df)) * 100, 2)

In [ ]:
# Drop rows with any missing value
df.dropna(inplace=True)
print(f'Dataset shape after removing missing values: {df.shape}')

In [ ]:
# Percentage of data retained
original_size = 74610
print(f'Percentage of remaining data: {round(len(df) / original_size * 100, 2)}%')

### 2.2 Check for Redundant Values in Categorical Columns

In [ ]:
def check_categorical_columns(data):
    cat_cols = data.select_dtypes(include='object').columns
    for col in cat_cols:
        print(f'\n{col}: {sorted(data[col].dropna().unique().tolist())}')

check_categorical_columns(df)

### 2.3 Drop Redundant Columns

In [ ]:
# 'Employee ID' is a unique identifier with no predictive value
df.drop(columns=['Employee ID'], inplace=True)
df.head()

## **3. Train-Validation Split**

In [ ]:
from sklearn.model_selection import train_test_split

# Features and target
X = df.drop(columns=['Attrition'])
y = df['Attrition']
print(f'Features shape : {X.shape}')
print(f'Target shape   : {y.shape}')
print(f'Target values  : {y.unique()}')

In [ ]:
# 70-30 split
X_train, X_validation, y_train, y_validation = train_test_split(
    X, y, test_size=0.3, random_state=42
)
print(f'Training set   : {X_train.shape[0]} rows')
print(f'Validation set : {X_validation.shape[0]} rows')

## **4. EDA on Training Data**

### 4.1 Univariate Analysis – Numerical Columns

In [ ]:
# Select numerical columns from training data
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
print('Numerical columns:', num_cols)

In [ ]:
# Distribution plots for all numerical columns
fig, axes = plt.subplots(len(num_cols), 1, figsize=(10, 4 * len(num_cols)))
for i, col in enumerate(num_cols):
    sns.histplot(X_train[col], ax=axes[i], kde=True, color='steelblue')
    axes[i].set_title(f'Distribution of {col}')
    axes[i].set_xlabel(col)
plt.tight_layout()
plt.show()

### 4.2 Correlation Analysis

In [ ]:
corr_matrix = X_train[num_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Matrix of Numerical Features (Training Set)')
plt.tight_layout()
plt.show()

### 4.3 Class Balance

In [ ]:
class_counts = y_train.value_counts()
plt.figure(figsize=(6, 4))
class_counts.plot(kind='bar', color=['steelblue', 'coral'], edgecolor='black')
plt.title('Class Balance in Training Set')
plt.xlabel('Attrition')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
print(class_counts)
print(f'\nClass proportions (%):\n{round(class_counts / class_counts.sum() * 100, 2)}')

### 4.4 Bivariate Analysis – Categorical vs Attrition

In [ ]:
cat_cols_train = X_train.select_dtypes(include='object').columns.tolist()
train_temp = X_train.copy()
train_temp['Attrition'] = y_train.values

for col in cat_cols_train:
    ct     = pd.crosstab(train_temp[col], train_temp['Attrition'])
    ct_pct = ct.div(ct.sum(axis=1), axis=0)
    ct_pct.plot(kind='bar', stacked=True, figsize=(10, 4),
                colormap='Set2', edgecolor='black')
    plt.title(f'{col} vs Attrition (Training Set)')
    plt.xlabel(col)
    plt.ylabel('Proportion')
    plt.xticks(rotation=45, ha='right')
    plt.legend(title='Attrition', loc='upper right')
    plt.tight_layout()
    plt.show()

## **5. EDA on Validation Data (Optional)**

In [ ]:
# Numerical columns in validation set
num_cols_val = X_validation.select_dtypes(include=['int64', 'float64']).columns.tolist()
print('Numerical columns (validation):', num_cols_val)

fig, axes = plt.subplots(len(num_cols_val), 1, figsize=(10, 4 * len(num_cols_val)))
for i, col in enumerate(num_cols_val):
    sns.histplot(X_validation[col], ax=axes[i], kde=True, color='coral')
    axes[i].set_title(f'Distribution of {col} (Validation)')
    axes[i].set_xlabel(col)
plt.tight_layout()
plt.show()

In [ ]:
corr_val = X_validation[num_cols_val].corr()
plt.figure(figsize=(12, 10))
sns.heatmap(corr_val, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Matrix of Numerical Features (Validation Set)')
plt.tight_layout()
plt.show()

In [ ]:
val_class_counts = y_validation.value_counts()
plt.figure(figsize=(6, 4))
val_class_counts.plot(kind='bar', color=['steelblue', 'coral'], edgecolor='black')
plt.title('Class Balance in Validation Set')
plt.xlabel('Attrition')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
print(val_class_counts)

In [ ]:
cat_cols_val = X_validation.select_dtypes(include='object').columns.tolist()
val_temp = X_validation.copy()
val_temp['Attrition'] = y_validation.values

for col in cat_cols_val:
    ct     = pd.crosstab(val_temp[col], val_temp['Attrition'])
    ct_pct = ct.div(ct.sum(axis=1), axis=0)
    ct_pct.plot(kind='bar', stacked=True, figsize=(10, 4),
                colormap='Set2', edgecolor='black')
    plt.title(f'{col} vs Attrition (Validation Set)')
    plt.xlabel(col)
    plt.ylabel('Proportion')
    plt.xticks(rotation=45, ha='right')
    plt.legend(title='Attrition', loc='upper right')
    plt.tight_layout()
    plt.show()

## **6. Feature Engineering**

### 6.1 Dummy Variable Creation

In [ ]:
# Identify categorical columns
cat_cols = X_train.select_dtypes(include='object').columns.tolist()
print('Categorical columns requiring dummy variables:')
for c in cat_cols:
    print(f'  {c}: {X_train[c].nunique()} unique values')

In [ ]:
# Create dummy variables for training set
X_train_dummies = pd.get_dummies(X_train[cat_cols], drop_first=True, dtype=int)

# Numerical columns (already scaled below, so just select them here)
num_cols_train = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Concatenate numerical + dummies
X_train = pd.concat([X_train[num_cols_train].reset_index(drop=True),
                     X_train_dummies.reset_index(drop=True)], axis=1)
print(f'X_train shape after dummy encoding: {X_train.shape}')

In [ ]:
# Create dummy variables for validation set (use same columns as training)
X_validation_dummies = pd.get_dummies(X_validation[cat_cols], drop_first=True, dtype=int)

num_cols_val2 = X_validation.select_dtypes(include=['int64', 'float64']).columns.tolist()
X_validation = pd.concat([X_validation[num_cols_val2].reset_index(drop=True),
                           X_validation_dummies.reset_index(drop=True)], axis=1)

# Align validation columns to training columns (fill missing with 0)
X_validation = X_validation.reindex(columns=X_train.columns, fill_value=0)
print(f'X_validation shape after dummy encoding: {X_validation.shape}')

In [ ]:
# Encode target: 1 = Stayed, 0 = Left
y_train_dummies      = pd.get_dummies(y_train,      dtype=int)
y_validation_dummies = pd.get_dummies(y_validation, dtype=int)

y_train_final      = y_train_dummies['Stayed'].reset_index(drop=True)
y_validation_final = y_validation_dummies['Stayed'].reset_index(drop=True)

print(f'y_train_final value counts:\n{y_train_final.value_counts()}')
print(f'\ny_validation_final value counts:\n{y_validation_final.value_counts()}')

### 6.2 Feature Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

# Scale numerical features only
num_features = [c for c in num_cols_train if c in X_train.columns]
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_train_scaled[num_features] = scaler.fit_transform(X_train[num_features])

X_validation_scaled = X_validation.copy()
X_validation_scaled[num_features] = scaler.transform(X_validation[num_features])

print(f'X_train_scaled shape     : {X_train_scaled.shape}')
print(f'X_validation_scaled shape: {X_validation_scaled.shape}')

## **7. Model Building**

### 7.1 Feature Selection with RFE

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE

logreg = LogisticRegression(max_iter=1000, random_state=42)
rfe    = RFE(logreg, n_features_to_select=15)
rfe.fit(X_train_scaled, y_train_final)

# Display selected features
rfe_df = pd.DataFrame({
    'Feature' : X_train_scaled.columns,
    'Selected': rfe.support_,
    'Rank'    : rfe.ranking_
}).sort_values('Rank')
print(rfe_df[rfe_df['Selected']].to_string(index=False))

In [ ]:
# Store selected feature names
col = X_train_scaled.columns[rfe.support_].tolist()
print(f'\nSelected {len(col)} features:')
print(col)

### 7.2 Build Logistic Regression with statsmodels

> **FIX:** We import directly from sub-modules (`statsmodels.tools.tools` and `statsmodels.discrete.discrete_model`) instead of `statsmodels.api`. The `api` module tries to load `statsmodels.graphics.tsaplots` which triggers a `deprecate_kwarg()` bug in statsmodels < 0.14. This approach works on any version.

In [ ]:
# ── statsmodels fix ───────────────────────────────────────────────────────────
# Import directly from sub-modules to avoid the broken graphics import chain.
from statsmodels.tools.tools               import add_constant as sm_add_constant
from statsmodels.discrete.discrete_model   import Logit        as sm_Logit
from statsmodels.stats.outliers_influence  import variance_inflation_factor

# Thin namespace so the rest of the notebook can use 'sm.add_constant' / 'sm.Logit'
class sm:
    add_constant = staticmethod(sm_add_constant)
    Logit        = sm_Logit

print('statsmodels sub-modules loaded successfully (no api import needed).')

In [ ]:
# Select RFE-chosen features from training set
X_train_rfe = X_train_scaled[col].reset_index(drop=True)
print(f'X_train_rfe shape: {X_train_rfe.shape}')
X_train_rfe.head()

In [ ]:
# Add constant (intercept) to training set
X_train_sm = sm.add_constant(X_train_rfe)
print(f'X_train_sm shape: {X_train_sm.shape}')

In [ ]:
# Fit Logistic Regression model
logit_model = sm.Logit(y_train_final, X_train_sm)
result      = logit_model.fit(disp=0)
print(result.summary2())

### 7.3 VIF Check

In [ ]:
vif_data = pd.DataFrame()
vif_data['Feature'] = X_train_sm.columns
vif_data['VIF']     = [
    variance_inflation_factor(X_train_sm.values, i)
    for i in range(X_train_sm.shape[1])
]
vif_data = vif_data.sort_values('VIF', ascending=False).reset_index(drop=True)
print(vif_data.to_string(index=False))

### 7.4 Training Set Predictions

In [ ]:
# Predict probabilities on training set
y_train_pred = result.predict(X_train_sm)
print('Sample predicted probabilities:', y_train_pred.values[:5])

# Reshape to array
y_train_pred = np.array(y_train_pred)
print(f'Shape: {y_train_pred.shape}')

In [ ]:
# Build prediction DataFrame
y_train_pred_df = pd.DataFrame({
    'Stayed'        : y_train_final.values,
    'Predicted_Prob': y_train_pred
})

# Default threshold: 0.5
y_train_pred_df['Predicted'] = (y_train_pred_df['Predicted_Prob'] >= 0.5).astype(int)
y_train_pred_df.head()

### 7.5 Training Set Evaluation

In [ ]:
from sklearn import metrics

train_accuracy = metrics.accuracy_score(y_train_pred_df['Stayed'], y_train_pred_df['Predicted'])
print(f'Training Accuracy: {round(train_accuracy * 100, 2)}%')

In [ ]:
train_conf_matrix = metrics.confusion_matrix(y_train_pred_df['Stayed'], y_train_pred_df['Predicted'])
print('Confusion Matrix (Training Set):\n', train_conf_matrix)

tn_train, fp_train, fn_train, tp_train = train_conf_matrix.ravel()
print(f'True Positive  (TP): {tp_train}')
print(f'True Negative  (TN): {tn_train}')
print(f'False Positive (FP): {fp_train}')
print(f'False Negative (FN): {fn_train}')

In [ ]:
train_sensitivity = tp_train / (tp_train + fn_train)
train_specificity = tn_train / (tn_train + fp_train)
train_precision   = tp_train / (tp_train + fp_train)
train_recall      = tp_train / (tp_train + fn_train)

print(f'Sensitivity (Recall): {round(train_sensitivity, 4)}')
print(f'Specificity         : {round(train_specificity, 4)}')
print(f'Precision           : {round(train_precision,   4)}')
print(f'Recall              : {round(train_recall,      4)}')

### 7.6 ROC Curve

In [ ]:
def draw_roc(actual, probs):
    fpr, tpr, thresholds = metrics.roc_curve(actual, probs, drop_intermediate=False)
    auc_score = metrics.roc_auc_score(actual, probs)
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2,
             label=f'ROC curve (AUC = {auc_score:.4f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=1,
             linestyle='--', label='Random classifier')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')
    plt.legend(loc='lower right')
    plt.grid(True, alpha=0.4)
    plt.tight_layout()
    plt.show()
    print(f'AUC Score: {auc_score:.4f}')
    return fpr, tpr, thresholds

fpr_train, tpr_train, roc_thresholds_train = draw_roc(
    y_train_pred_df['Stayed'], y_train_pred_df['Predicted_Prob']
)

### 7.7 Optimal Cutoff – Sensitivity / Specificity Tradeoff

In [ ]:
# Predict at various probability cutoffs
cutoffs = np.arange(0.0, 1.0, 0.1)
for cutoff in cutoffs:
    col_label = f'Pred_{round(cutoff, 1)}'
    y_train_pred_df[col_label] = (y_train_pred_df['Predicted_Prob'] >= cutoff).astype(int)
y_train_pred_df.head()

In [ ]:
# Accuracy, Sensitivity, Specificity at each cutoff
rows = []
for cutoff in cutoffs:
    col_label = f'Pred_{round(cutoff, 1)}'
    cm = metrics.confusion_matrix(y_train_pred_df['Stayed'], y_train_pred_df[col_label])
    tn_c, fp_c, fn_c, tp_c = cm.ravel()
    acc  = (tp_c + tn_c) / (tp_c + tn_c + fp_c + fn_c)
    sens = tp_c / (tp_c + fn_c) if (tp_c + fn_c) > 0 else 0
    spec = tn_c / (tn_c + fp_c) if (tn_c + fp_c) > 0 else 0
    rows.append({'Cutoff': round(cutoff, 1), 'Accuracy': round(acc, 4),
                 'Sensitivity': round(sens, 4), 'Specificity': round(spec, 4)})

cutoff_df = pd.DataFrame(rows)
print(cutoff_df.to_string(index=False))

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(cutoff_df['Cutoff'], cutoff_df['Accuracy'],    label='Accuracy',    marker='o', linewidth=2)
plt.plot(cutoff_df['Cutoff'], cutoff_df['Sensitivity'], label='Sensitivity', marker='s', linewidth=2)
plt.plot(cutoff_df['Cutoff'], cutoff_df['Specificity'], label='Specificity', marker='^', linewidth=2)
plt.xlabel('Probability Cutoff')
plt.ylabel('Score')
plt.title('Accuracy, Sensitivity, and Specificity vs Cutoff (Training Data)')
plt.legend()
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# Optimal cutoff = threshold where |Sensitivity - Specificity| is minimised
diff_train     = (cutoff_df['Sensitivity'] - cutoff_df['Specificity']).abs()
optimal_cutoff = float(cutoff_df.loc[diff_train.idxmin(), 'Cutoff'])
print(f'Optimal cutoff (training): {optimal_cutoff}')

y_train_pred_df['final_predicted'] = (
    y_train_pred_df['Predicted_Prob'] >= optimal_cutoff).astype(int)

In [ ]:
opt_train_acc = metrics.accuracy_score(
    y_train_pred_df['Stayed'], y_train_pred_df['final_predicted'])
print(f'Accuracy at optimal cutoff ({optimal_cutoff}): {round(opt_train_acc * 100, 2)}%')

opt_train_cm = metrics.confusion_matrix(
    y_train_pred_df['Stayed'], y_train_pred_df['final_predicted'])
print('Confusion Matrix at optimal cutoff:\n', opt_train_cm)

tn_opt, fp_opt, fn_opt, tp_opt = opt_train_cm.ravel()
print(f'True Positive  (TP): {tp_opt}')
print(f'True Negative  (TN): {tn_opt}')
print(f'False Positive (FP): {fp_opt}')
print(f'False Negative (FN): {fn_opt}')

opt_train_sens = tp_opt / (tp_opt + fn_opt)
opt_train_spec = tn_opt / (tn_opt + fp_opt)
opt_train_prec = tp_opt / (tp_opt + fp_opt)
opt_train_rec  = tp_opt / (tp_opt + fn_opt)

print(f'Sensitivity : {round(opt_train_sens, 4)}')
print(f'Specificity : {round(opt_train_spec, 4)}')
print(f'Precision   : {round(opt_train_prec, 4)}')
print(f'Recall      : {round(opt_train_rec,  4)}')

### 7.8 Precision-Recall Curve (Training)

In [ ]:
from sklearn.metrics import precision_recall_curve

y_actual_train    = y_train_pred_df['Stayed']
y_pred_prob_train = y_train_pred_df['Predicted_Prob']

precision_curve, recall_curve, pr_thresholds = precision_recall_curve(
    y_actual_train, y_pred_prob_train)

plt.figure(figsize=(8, 6))
plt.plot(recall_curve, precision_curve, marker='.', linewidth=2,
         color='darkorange', label='Logistic Regression')
plt.xlabel('Recall',    fontsize=12)
plt.ylabel('Precision', fontsize=12)
plt.title('Precision-Recall Curve (Training Set)', fontsize=13)
plt.legend()
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

## **8. Prediction and Model Evaluation (Validation Set)**

### 8.1 Make Predictions on Validation Set

In [ ]:
# ── FIX: use the same 'col' list (from RFE) – not the loop variable 'col' ────
# Select RFE-chosen features from validation set
X_validation_rfe = X_validation_scaled[col].reset_index(drop=True)
print(f'X_validation_rfe shape: {X_validation_rfe.shape}')

In [ ]:
# Add constant (same as training)
X_validation_sm = sm.add_constant(X_validation_rfe)

# ── FIX: ensure validation has the same columns as training (const may differ) ─
# Reindex to match training columns exactly
X_validation_sm = X_validation_sm.reindex(columns=X_train_sm.columns, fill_value=0)
print(f'X_validation_sm shape: {X_validation_sm.shape}')

In [ ]:
# Predict on validation set
y_validation_pred = result.predict(X_validation_sm)
print('Sample validation predictions:', y_validation_pred.values[:5])

In [ ]:
# Build validation prediction DataFrame
predicted_probability = pd.DataFrame(y_validation_pred.values, columns=['Predicted_Prob'])
actual                = pd.DataFrame(y_validation_final.values, columns=['Stayed'])

y_validation_df = pd.concat([actual.reset_index(drop=True),
                              predicted_probability.reset_index(drop=True)], axis=1)

# Apply optimal cutoff
y_validation_df['final_prediction'] = (
    y_validation_df['Predicted_Prob'] >= optimal_cutoff).astype(int)

y_validation_df.head()

### 8.2 Accuracy

In [ ]:
val_accuracy = metrics.accuracy_score(
    y_validation_df['Stayed'], y_validation_df['final_prediction'])
print(f'Validation Accuracy: {round(val_accuracy * 100, 2)}%')

### 8.3 Confusion Matrix

In [ ]:
val_conf_matrix = metrics.confusion_matrix(
    y_validation_df['Stayed'], y_validation_df['final_prediction'])
print('Validation Confusion Matrix:\n', val_conf_matrix)

tn_val, fp_val, fn_val, tp_val = val_conf_matrix.ravel()
print(f'True Positive  (TP): {tp_val}')
print(f'True Negative  (TN): {tn_val}')
print(f'False Positive (FP): {fp_val}')
print(f'False Negative (FN): {fn_val}')

### 8.4 Sensitivity & Specificity

In [ ]:
val_sensitivity = tp_val / (tp_val + fn_val)
print(f'Sensitivity (Recall): {round(val_sensitivity, 4)}')

In [ ]:
# Specificity at each threshold + at optimal cutoff
thresholds_spec = np.arange(0.0, 1.0, 0.1)
y_val_probs     = np.array(result.predict(X_validation_sm))

fpr_vals, tnr_vals = [], []
for thresh in thresholds_spec:
    y_pred_t = (y_val_probs >= thresh).astype(int)
    cm_t = metrics.confusion_matrix(y_validation_df['Stayed'], y_pred_t)
    tn_t, fp_t, fn_t, tp_t = cm_t.ravel()
    fpr_vals.append(fp_t / (fp_t + tn_t) if (fp_t + tn_t) > 0 else 0)
    tnr_vals.append(tn_t / (tn_t + fp_t) if (tn_t + fp_t) > 0 else 0)

tn_threshold_df = pd.DataFrame({
    'Threshold'         : np.round(thresholds_spec, 1),
    'FPR'               : np.round(fpr_vals, 4),
    'Specificity (TNR)' : np.round(tnr_vals, 4)
})
print(tn_threshold_df.to_string(index=False))

val_specificity = tn_val / (tn_val + fp_val)
val_fpr         = fp_val / (fp_val + tn_val)
print(f'\nSpecificity at optimal cutoff ({optimal_cutoff}): {round(val_specificity, 4)}')
print(f'False Positive Rate                             : {round(val_fpr, 4)}')

### 8.5 Precision & Recall

In [ ]:
from sklearn.metrics import precision_score, recall_score

thresholds_seq   = np.arange(0.0, 1.0, 0.1)
precision_scores = []
recall_scores    = []

for thresh in thresholds_seq:
    y_pred_t = (y_validation_df['Predicted_Prob'] >= thresh).astype(int)
    precision_scores.append(precision_score(y_validation_df['Stayed'], y_pred_t, zero_division=0))
    recall_scores.append(recall_score(y_validation_df['Stayed'],    y_pred_t, zero_division=0))

pr_threshold_df = pd.DataFrame({
    'Threshold': np.round(thresholds_seq, 1),
    'Precision': np.round(precision_scores, 4),
    'Recall'   : np.round(recall_scores,    4)
})
print(pr_threshold_df.to_string(index=False))

opt_prec = precision_score(y_validation_df['Stayed'], y_validation_df['final_prediction'], zero_division=0)
opt_rec  = recall_score(   y_validation_df['Stayed'], y_validation_df['final_prediction'], zero_division=0)
print(f'\nPrecision at optimal cutoff ({optimal_cutoff}): {round(opt_prec, 4)}')
print(f'Recall    at optimal cutoff ({optimal_cutoff}): {round(opt_rec,  4)}')

### 8.6 Intersection Graph – Optimal Threshold & Generalization Performance

In [ ]:
accuracy_vals       = []
sensitivity_vals_pl = []
specificity_vals_pl = []

for thresh in thresholds_seq:
    y_pred_t = (y_validation_df['Predicted_Prob'] >= thresh).astype(int)
    cm_t = metrics.confusion_matrix(y_validation_df['Stayed'], y_pred_t)
    tn_t, fp_t, fn_t, tp_t = cm_t.ravel()
    total = tp_t + tn_t + fp_t + fn_t
    accuracy_vals.append((tp_t + tn_t) / total)
    sensitivity_vals_pl.append(tp_t / (tp_t + fn_t) if (tp_t + fn_t) > 0 else 0.0)
    specificity_vals_pl.append(tn_t / (tn_t + fp_t) if (tn_t + fp_t) > 0 else 0.0)

# Intersection plot
plt.figure(figsize=(12, 7))
plt.plot(thresholds_seq, accuracy_vals,       label='Accuracy',           marker='o', linewidth=2.5, color='steelblue')
plt.plot(thresholds_seq, sensitivity_vals_pl, label='Sensitivity/Recall', marker='s', linewidth=2.5, color='darkorange')
plt.plot(thresholds_seq, specificity_vals_pl, label='Specificity',        marker='^', linewidth=2.5, color='forestgreen')

diff_cross   = [abs(s - sp) for s, sp in zip(sensitivity_vals_pl, specificity_vals_pl)]
crossing_idx = int(np.argmin(diff_cross))
crossing_x   = float(thresholds_seq[crossing_idx])
crossing_y   = (sensitivity_vals_pl[crossing_idx] + specificity_vals_pl[crossing_idx]) / 2.0

plt.axvline(x=crossing_x, color='red', linestyle='--', linewidth=1.5,
            label=f'Optimal threshold = {crossing_x:.1f}')
plt.scatter([crossing_x], [crossing_y], color='red', s=120, zorder=6)
plt.annotate(
    f'  ({crossing_x:.1f},\n  {crossing_y:.3f})',
    xy=(crossing_x, crossing_y),
    xytext=(crossing_x + 0.04, crossing_y - 0.06),
    fontsize=10, color='red',
    arrowprops=dict(arrowstyle='->', color='red')
)
plt.xlabel('Threshold',  fontsize=13)
plt.ylabel('Score',      fontsize=13)
plt.title('Accuracy, Sensitivity/Recall, and Specificity vs Threshold\n(Validation Set)', fontsize=14)
plt.legend(fontsize=11, loc='center left')
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

print('Optimal Crossing Coordinate:')
print(f'  Threshold          : {crossing_x:.2f}')
print(f'  Accuracy           : {accuracy_vals[crossing_idx]:.4f}')
print(f'  Sensitivity/Recall : {sensitivity_vals_pl[crossing_idx]:.4f}')
print(f'  Specificity        : {specificity_vals_pl[crossing_idx]:.4f}')

deployment_threshold = crossing_x
print(f'\nDeployment Criterion (optimal balance threshold): {deployment_threshold:.2f}')

In [ ]:
# Generalization performance on held-out validation set
y_gen_probs = np.array(result.predict(X_validation_sm))
y_gen_pred  = (y_gen_probs >= deployment_threshold).astype(int)

gen_cm = metrics.confusion_matrix(y_validation_final, y_gen_pred)
tn_g, fp_g, fn_g, tp_g = gen_cm.ravel()

gen_accuracy    = metrics.accuracy_score(   y_validation_final, y_gen_pred)
gen_sensitivity = tp_g / (tp_g + fn_g) if (tp_g + fn_g) > 0 else 0.0
gen_specificity = tn_g / (tn_g + fp_g) if (tn_g + fp_g) > 0 else 0.0
gen_precision   = metrics.precision_score(  y_validation_final, y_gen_pred, zero_division=0)
gen_recall      = metrics.recall_score(     y_validation_final, y_gen_pred, zero_division=0)

print('Generalization Performance on Out-of-Sample Validation Set:')
print(f'  Deployment Threshold  : {deployment_threshold:.2f}')
print(f'  Accuracy              : {round(gen_accuracy,    4)}')
print(f'  Sensitivity (Recall)  : {round(gen_sensitivity, 4)}')
print(f'  Specificity           : {round(gen_specificity, 4)}')
print(f'  Precision             : {round(gen_precision,   4)}')
print(f'  Recall                : {round(gen_recall,      4)}')

## Conclusion

This notebook built a complete Logistic Regression pipeline for predicting employee retention:

1. **Data Cleaning** — dropped rows with missing values (~5.3% of data), removed the non-predictive `Employee ID`.
2. **EDA** — explored distributions, correlations, class balance, and bivariate relationships.
3. **Feature Engineering** — dummy-encoded 15 categorical columns, applied `StandardScaler` to 7 numerical columns.
4. **RFE** — selected the top 15 features using `LogisticRegression` as the estimator.
5. **statsmodels Logit** — fit with statistical summaries (p-values, coefficients, VIF).
6. **Threshold Optimisation** — found the optimal cutoff where |Sensitivity − Specificity| is minimised.
7. **Validation** — evaluated accuracy, sensitivity, specificity, precision, recall, and plotted ROC + Precision-Recall curves.